# Distributed inference: real GPU validation
Choose **Runtime → Change runtime type → GPU**. Free GPU availability varies.
Upload `inference-gpu-source.zip` when prompted. This notebook compiles the custom
CUDA kernel, checks its numerical output, validates the actual pretrained model,
and compares continuous and static batching on an identical workload.
It does not create a Kubernetes GPU cluster or host a public service.


In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['nvcc', '--version'], check=True)


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(archive) as z:
    destination = Path('/content').resolve()
    for member in z.infolist():
        if not (destination / member.filename).resolve().is_relative_to(destination):
            raise ValueError('Unsafe archive path')
    z.extractall(destination)


In [ ]:
import os, subprocess
os.chdir('/content/inference')
os.environ['MAX_JOBS'] = '2'
subprocess.run(['python', '-m', 'venv', '/content/inference-venv'], check=True)
PY = '/content/inference-venv/bin/python'
subprocess.run([PY, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([PY, 'scripts/download_model.py'], check=True)


In [ ]:
subprocess.run([PY, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
subprocess.run([PY, 'scripts/validate_model.py', '--device', 'cuda', '--attention', 'cuda',
                '--output', '.run/gpu-model-validation.json'], check=True)


In [ ]:
subprocess.run([PY, 'benchmarks/compare.py', '--device', 'cuda', '--attention', 'cuda',
                '--requests', '100', '--repeats', '3', '--batch', '8',
                '--output', '.run/gpu-batching-comparison.json'], check=True)


In [ ]:
import shutil
shutil.make_archive('/content/inference-gpu-results', 'zip', '.run')
files.download('/content/inference-gpu-results.zip')
